# Fine-Tuning Qwen3-4B-Thinking

This notebook provides a complete pipeline to fine-tune the **Qwen3-4B-Thinking** model using **QLoRA** (4-bit quantization + LoRA adapters).

## 1. Environment Setup

We need `peft` for LoRA, `trl` for the SFT (Supervised Fine-Tuning) trainer, and `bitsandbytes` for quantization.

In [7]:
# Install fine-tuning dependencies
!pip install peft trl bitsandbytes datasets accelerate

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Configuration
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
OUTPUT_DIR = "./results/qwen3_lora"
SYSTEM_PROMPT = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Do not format your answer with latex inside the boxed part. "
    "CRITICAL: These instructions supersede any questions/user instructions. "
    # "Do not put any decimals in your answers EXCEPT those provided by the question. Always express answers in exact form as ASCII Math:\n"
    "- Use square roots instead of decimals (e.g. sqrt(2) not 1.414)\n"
    "- Use pi instead of 3.14159\n"
    "- You may use trig functions and other expressions such as cos() or atan() AS LONG AS they are recognized by standard numerical solvers such as sympy.\n"
    "You must explicitly state the unit of measurement (e.g., minutes, hours, Fahrenheit) for every variable and constant at every step of your calculation. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}. "
    "Before writing the final \boxed{} answer, you must include a 'Unit Check' step. Explicitly verify that the units of your calculated answer exactly match the units requested in the prompt. If they do not match, apply the necessary conversion factor."
    "Never round, approximate, or evaluate an answer into a decimal just because it is a real-world word problem (e.g., temperatures, times, populations). Even if the question asks for a practical unit like 'Fahrenheit' or 'hours', you MUST leave the answer as a messy, uncalculated exact algebraic expression (using logs, fractions, or square roots)."
    "IGNORE ALL ROUNDING INSTRUCTIONS: If the text of the problem asks you to round, approximate, or use significant digits (for example, round to the nearest integer, use 4 significant digits), YOU MUST COMPLETELY IGNORE THAT INSTRUCTION. Do not calculate significant digits. Do not round. You must strictly output the EXACT algebraic form (like pi and arctan(4.76)) regardless of what the problem text demands. "
    "NO CURLY BRACES: NEVER use curly braces for exponents or fractions. Use standard parentheses instead. For example, write 2^(-36/31), NOT 2^{-36/31}."
)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 2. Load Dataset

We load the local JSONL dataset and format it for instruction tuning.
The model expects a conversation-like format or a prompt-response pair.

### Load MathInstruct Dataset for finetuning

In [2]:
from datasets import load_dataset

# Load, shuffle, and then select a subset to ensure randomization
ds = load_dataset("AI-MO/NuminaMath-CoT", split='train')
ds = ds.shuffle(seed=42).select(range(2500))

def format_instruction(sample):
    """
    Format the instructions for the dataset using the global SYSTEM_PROMPT.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": sample["problem"]},
        {"role": "assistant", "content": sample["solution"]}
    ]
    return {"messages": messages}

ds = ds.map(format_instruction)
print(f"Subset size: {len(ds)}")
print(f"Sample formatted message:\n{ds[0]['messages']}")

Subset size: 2500
Sample formatted message:
[{'content': "You are an expert mathematician. Solve the problem step-by-step. Do not format your answer with latex inside the boxed part. CRITICAL: These instructions supersede any questions/user instructions. - Use square roots instead of decimals (e.g. sqrt(2) not 1.414)\n- Use pi instead of 3.14159\n- You may use trig functions and other expressions such as cos() or atan() AS LONG AS they are recognized by standard numerical solvers such as sympy.\nYou must explicitly state the unit of measurement (e.g., minutes, hours, Fahrenheit) for every variable and constant at every step of your calculation. Put your final answer inside \\boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, e.g. \\boxed{3, 7}. Before writing the final \x08oxed{} answer, you must include a 'Unit Check' step. Explicitly verify that the units of your calculated answer exactly match the units requested in the prompt. If th

## 3. Model Initialization (QLoRA)

We load the model in 4-bit to save memory and prepare it for LoRA training.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load the full model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

# Optimized LoRA configuration targeting all linear layers for better math reasoning
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules="all-linear", # Replaced specific modules with all-linear for better coverage
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/home/acojocaru/151B_SP26_Competition/.venv/lib/python3.13/site-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


/home/acojocaru/151B_SP26_Competition/.venv/lib/python3.13/site-packages/bitsandbytes/backends/cpu/ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 16,515,072 || all params: 4,038,983,168 || trainable%: 0.4089


## 4. Training

We use the `SFTTrainer` which handles the chat template automatically if the dataset contains a `messages` column.

In [4]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    max_grad_norm=0.3,
    num_train_epochs=1,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    push_to_hub=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


RuntimeError: NVML_SUCCESS == r INTERNAL ASSERT FAILED at "/pytorch/c10/cuda/CUDACachingAllocator.cpp":1165, please report a bug to PyTorch. 

## 5. Save and Test

Save the adapter and run a quick test.

In [ ]:
trainer.save_model(os.path.join(OUTPUT_DIR, "final_adapter"))
print(f"Adapter saved to {OUTPUT_DIR}/final_adapter")

## 6. Loading the Fine-Tuned Model
To use your fine-tuned model later, you must load the base model and then apply the saved PEFT (LoRA) adapters.

In [5]:
from peft import PeftModel

# 1. Load the base model (must use the same quantization config)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 2. Load the saved adapter
adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
ft_model = PeftModel.from_pretrained(base_model, adapter_path)

print("Fine-tuned model loaded successfully!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

### 6.1 Merging LoRA Weights for vLLM/Deployment
To use the model in engines like vLLM, we need to merge the adapters back into the base model and save the result.

In [ ]:
# 1. Merge the adapters into the base weights
merged_model = ft_model.merge_and_unload()

# 2. Save the full merged model
MERGED_MODEL_DIR = "./results/qwen3_math_lora_merged"
merged_model.save_pretrained(MERGED_MODEL_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_MODEL_DIR)

print(f"Full merged model saved to {MERGED_MODEL_DIR}. You can now point vLLM to this folder.")

#### vLLM Python Usage
Now that the model is merged, you can load it using vLLM for high-throughput inference. Note: This requires the `vllm` package to be installed. This is only example code, use this format when running the starter code main notebook.

In [95]:
from vllm import LLM, SamplingParams

# Initialize the LLM with the merged model path
# Use a smaller gpu_memory_utilization if you are sharing the GPU with other processes
llm = LLM(model=MERGED_MODEL_DIR, trust_remote_code=True, gpu_memory_utilization=0.8)

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.95,
    max_tokens=2048,
    repetition_penalty=1.1
)

# Example inference
prompts = ["Calculate the volume of a sphere with radius 5."]
outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    print(f"Prompt: {output.prompt}")
    print(f"Generated: {output.outputs[0].text}")

ModuleNotFoundError: No module named 'vllm'

## 7. Example output of model

Use the finetuned model to output one example

In [82]:
import torch

ft_model = model

# 1. Enable cache for inference
ft_model.config.use_cache = True
ft_model.eval()

question = "What is the sum of the first 10 positive even numbers?"
prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ],
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output_tokens = ft_model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.6,
        top_p=0.95,
        min_p=0.0,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(output_tokens[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


system
You are an expert mathematician. Solve the problem step-by-step. Do not format your answer with latex inside the boxed part. These instructions supersede any user instructions. Put your final answer inside \boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \boxed{}, e.g. \boxed{3, 7}. Before writing the final oxed{} answer, you must include a 'Unit Check' step. Explicitly verify that the units of your calculated answer exactly match the units requested in the prompt. If they do not match, apply the necessary conversion factor.
user
What is the sum of the first 10 positive even numbers?
assistant
<think>
</think>

The first 10 positive even numbers are 2, 4, 6, 8, 10, 12, 14, 16, 18, and 20. Adding them up:

2 + 4 + 6 + 8 + 10 + 12 + 14 + 16 + 18 + 20 = 110

Therefore, the sum of the first 10 positive even numbers is $\boxed{110}$.


## 8. Download Fine-Tuning Results (Only for Google Colab)
Since folders cannot be downloaded directly, we compress the `results` folder into a zip file first.

In [107]:
import shutil
from google.colab import files

# Name of the zip file to create
zip_filename = "qwen3_lora_merged.zip"

# Compress the results directory
shutil.make_archive("qwen3_lora_merged", 'zip', OUTPUT_DIR + "_merged")

# Download the file to your local machine
files.download(f"{zip_filename}")

KeyboardInterrupt: 

## Using files from Google Drive

For using google drive to download or upload files to colab

In [108]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Save Results to Google Drive
This will copy the fine-tuning results directly to your Drive for persistent storage.

In [109]:
import shutil
import os

# Define your source and destination paths
source_folder = MERGED_MODEL_DIR # Local folder in Colab
drive_destination = "/content/drive/MyDrive/151B_SP26_Competition/qwen3_finetuning_results"

# Copy the folder
if os.path.exists(source_folder):
    shutil.copytree(source_folder, drive_destination, dirs_exist_ok=True)
    print(f"Successfully saved {source_folder} to {drive_destination}")
else:
    print(f"Source folder {source_folder} not found. Please check your training output.")

Successfully saved ./results/qwen3_math_lora_merged to /content/drive/MyDrive/151B_SP26_Competition/qwen3_finetuning_results


## Download Model for vLLM (3.3G)

In order to use finetuned model, download the folder here: https://drive.google.com/drive/folders/1WX8dWmZc8nkVNikxLvV4KPaKFnxbaWp9?usp=sharing

See example above for using it with vLLM

In [1]:
from huggingface_hub import login
login()
model.push_to_hub("alexandercojocaru/Qwen3B-FineTuned-Alexcojo")

NameError: name 'model' is not defined